# Upscale — data preprocessing

Turns source frames into the `(low resolution, high resolution)` patch pairs the
upscaling model trains on, and writes them next to a manifest that
`upscale_dummy.ipynb` reads back.

The pipeline is four steps:

1. **Collect** frames from the public datasets `datasets/download.py` fetched — photos
   (DIV2K, Flickr2K) and YouTube sequences, desktop and game footage among them — read
   through `sources.py`, plus any screenshots of your own in `datasets/raw/captures/`. With none of
   that on disk it synthesises desktop-like frames instead, so the notebook runs end to
   end before a dataset exists.
2. **Degrade** each frame the way the stream does (`degrade.py`): downscale, 4:2:0,
   H.264 at a constant bitrate, decode. Every source draws its own filter and bitrate, and
   the draw is recorded in the manifest.
3. **Tile** both into aligned patches, dropping the near-flat ones — a desktop frame is
   mostly empty background, and a model trained on it learns to copy flat colour.
4. **Save** four `uint8` arrays (train/val × lr/hr) plus `manifest.json`, into
   `datasets/upscale/` — beside the `raw/` they were made from.

The split is by *source* — an image, or a whole YouTube video — never by patch:
neighbouring patches of one frame overlap, and frames of one video nearly repeat, so
splitting later would put near-copies of the training data into validation and every
metric downstream would read high. The datasets come split already (`train.txt` /
`test.txt`), and their test half is the validation set here.

In [ ]:
"""Configuration — every knob the notebook has."""

import os
from pathlib import Path

# The datasets folder - DATASETS_DIR, else datasets/ at the repo root, relative to this
# notebook's folder (upscale/) - holds the downloads in raw/ and this problem's output in upscale/.
DATASETS_ROOT = Path(os.environ.get("DATASETS_DIR", "../../datasets"))
CAPTURE_DIR = DATASETS_ROOT / "raw" / "captures"   # your own screenshots, used as they are
OUT_DIR = DATASETS_ROOT / "upscale"
SYNTHETIC_DIR = OUT_DIR / "synthetic"              # synthesised desktop frames

SCALE = 2                      # upscaling factor the model will be trained for
PATCH_HR = 128                 # HR patch edge in pixels; LR patch is PATCH_HR // SCALE
STRIDE_HR = 96                 # tiling stride in HR pixels (below PATCH_HR they overlap)
VAR_MIN = 24.0                 # drop a patch whose luma variance is below this
MAX_PATCHES_PER_FRAME = 32     # cap, so one large frame cannot dominate the set
VAL_FRACTION = 0.2             # share of the captures held out — the datasets bring their own split
SEED = 0

# How many files of each dataset to use: "train" from its train.txt, "val" from its
# test.txt, None for all of them. A dataset that is not on disk is skipped with a note.
# Patches add up fast: 100 sources x 32 patches is ~160 MB of HR patches.
DATASETS = {
    "div2k": {"train": 100, "val": 25},
    "flickr2k": {"train": 100},
    "yt8m-videos": {"train": 100, "val": 25},
}
FRAMES_PER_SEQUENCE = 2        # frames taken from each YouTube sequence, evenly spaced

# "stream": every source takes its own random trip through the stream (degrade.sample) —
# filter, bitrate, how long a still has been on screen. "clean": a bicubic downscale and
# 4:2:0 only, no codec — the pairs this notebook made before, for comparison.
DEGRADATION = "stream"

# Synthetic desktop frames: always written, as the whole frame the model notebooks upscale
# for their demos, and trained on only when there is no dataset and no capture.
SYNTHESISE_IF_EMPTY = True
SYNTHETIC_FRAMES = 12
SYNTHETIC_SIZE = (960, 600)

assert PATCH_HR % SCALE == 0, "PATCH_HR must be a multiple of SCALE"
PATCH_LR = PATCH_HR // SCALE

In [ ]:
"""Imports, folders, and the RNG everything below draws from."""

import json
import time

import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

import degrade
import sources

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

rng = np.random.default_rng(SEED)

CAPTURE_DIR.mkdir(parents=True, exist_ok=True)
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)

print("datasets  ", sources.ROOT)
print("captures  ", CAPTURE_DIR.resolve())
print("processed ", OUT_DIR.resolve())

## 1. Source frames

Three places, in this order:

- **the datasets** in `DATASETS`, as many files of each as asked for, drawn at random
  from its list. Fetch them with `uv run datasets/download.py div2k flickr2k` and
  `uv run datasets/download.py yt8m-videos --total 1000 --sequences 5`;
- **`datasets/raw/captures/`**, used as it is — screenshots, or frames pulled out of a recording. To
  fill it from one, desktop-streamer (checked out beside this repo) ships ffmpeg:

```
../../../desktop-streamer/src/client/native/win32-x64/libs/ffmpeg/ffmpeg -i capture.mp4 -vf fps=1 ../../datasets/raw/captures/frame_%04d.png
```

  One frame per second, because consecutive frames of a screen recording are nearly
  identical and a hundred of them teach the model no more than one of them does;
- **synthetic frames** in `datasets/upscale/synthetic/`, only if both of the above came up
  empty. They are written either way: the model notebooks upscale one whole as a demo.

The datasets are photographs and camera video, not desktops. They give the model
general structure; screen content of your own in `datasets/raw/captures/` is what teaches it text and
UI edges.

In [ ]:
"""Draw a synthetic desktop frame: flat chrome, text-like rows, one noisy region."""

def synthesise_frame(size, rng):
    width, height = size
    background = tuple(int(v) for v in rng.integers(20, 60, size=3))
    frame = Image.new("RGB", size, background)
    draw = ImageDraw.Draw(frame)

    # A soft vertical gradient, the kind a desktop wallpaper has.
    for y in range(height):
        shade = int(18 * y / height)
        draw.line([(0, y), (width, y)], fill=tuple(min(255, c + shade) for c in background))

    for _ in range(int(rng.integers(2, 5))):
        w = int(rng.integers(width // 4, width // 2))
        h = int(rng.integers(height // 4, height // 2))
        x = int(rng.integers(0, max(1, width - w)))
        y = int(rng.integers(0, max(1, height - h)))
        panel = tuple(int(v) for v in rng.integers(180, 250, size=3))
        accent = tuple(int(v) for v in rng.integers(40, 140, size=3))

        draw.rectangle([x, y, x + w, y + h], fill=panel, outline=accent, width=2)
        draw.rectangle([x, y, x + w, y + 22], fill=accent)          # title bar
        for i in range(3):                                          # window buttons
            draw.ellipse([x + w - 18 - i * 16, y + 6, x + w - 8 - i * 16, y + 16], fill=panel)

        # Text-like rows: the high frequency detail upscaling is actually judged on.
        ink = tuple(int(v) for v in rng.integers(0, 90, size=3))
        row = y + 34
        while row < y + h - 12:
            length = int(rng.integers(w // 4, max(w // 4 + 1, w - 24)))
            draw.rectangle([x + 12, row, x + 12 + length, row + 4], fill=ink)
            row += int(rng.integers(10, 18))

    # One block of noise stands in for photo or video content playing on the screen.
    nw, nh = width // 5, height // 5
    nx, ny = int(rng.integers(0, width - nw)), int(rng.integers(0, height - nh))
    frame.paste(Image.fromarray(rng.integers(0, 255, (nh, nw, 3), dtype=np.uint8)), (nx, ny))

    return frame


# One entry per source: where it is, which split it belongs to, and whether it is a
# single image or a sequence of frames.
items = []
for name, wanted in DATASETS.items():
    for split, count in wanted.items():
        try:
            paths = sources.files(name, "test" if split == "val" else "train")
        except FileNotFoundError as error:
            print(f"  {name}: not on disk, skipped ({error})")
            break
        if count is not None and count < len(paths):
            paths = [paths[i] for i in sorted(rng.choice(len(paths), count, replace=False))]
        kind = "sequence" if name in sources.SEQUENCES else "image"
        items += [{"dataset": name, "split": split, "path": p, "kind": kind} for p in paths]

# The synthetic frames are always on disk - the model notebooks upscale one whole as a demo -
# and drawn with their own generator, so making them or not leaves every other draw as it was.
synthetic = sorted(SYNTHETIC_DIR.glob("synthetic_*.png"))
if len(synthetic) < SYNTHETIC_FRAMES:
    synthetic_rng = np.random.default_rng(SEED)
    synthetic = [SYNTHETIC_DIR / f"synthetic_{i:04d}.png" for i in range(SYNTHETIC_FRAMES)]
    for path in synthetic:
        synthesise_frame(SYNTHETIC_SIZE, synthetic_rng).save(path)

# Your captures, split here since nothing has split them yet - or, with no dataset and no
# capture, the synthetic frames in their place.
own, label = sorted(p for p in CAPTURE_DIR.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES), "captures"
if not own and not items and SYNTHESISE_IF_EMPTY:
    print(f"no datasets and {CAPTURE_DIR} is empty — training on the {len(synthetic)} synthetic frames")
    own, label = synthetic, "synthetic"
order = rng.permutation(len(own))
n_val = max(1, round(len(own) * VAL_FRACTION)) if len(own) > 1 else 0
held_out = {own[i] for i in order[:n_val]}
items += [{"dataset": label, "split": "val" if p in held_out else "train", "path": p, "kind": "image"}
          for p in own]

if not items:
    raise RuntimeError(f"no datasets, no images in {CAPTURE_DIR}, and synthesis is off")

for split in ("train", "val"):
    counts = {}
    for item in items:
        if item["split"] == split:
            counts[item["dataset"]] = counts.get(item["dataset"], 0) + 1
    print(f"{split:5} {sum(counts.values()):4} sources  {counts}")

## 2. Degradation

`degrade.degrade` crops the frame to a multiple of `2 * SCALE` — that crop is the HR
target — and runs it through the stream: a downscale in RGB, 4:2:0, H.264 with the host's
libx264 line at a constant bitrate, and back. Cropping first is what keeps the two
aligned — a downscale of an odd width would round, and every patch pair after it would
sit half a pixel off.

With `DEGRADATION = "stream"` each source draws its own trip (`degrade.sample`): the
scaler the host would use, a bitrate in bits per pixel per frame from the host's 200
kbit/s floor to its 8 Mbit/s default, and — for a single image — how many frames it has
been on screen, from the keyframe (the worst picture a stream shows) to a still that has
settled. A YouTube sequence is encoded as the video it is, so its frames carry real
inter-frame artifacts, and `FRAMES_PER_SEQUENCE` of them are kept. The draw is written
into the manifest per source, so a result can later be split by bitrate or filter.

In [ ]:
"""Load one source, take it through the stream, and keep the frames it contributes."""

def degradation_for(fps):
    if DEGRADATION == "clean":
        return degrade.Degradation(scale=SCALE, filter="bicubic", bpp=None, fps=fps)
    return degrade.sample(rng, scale=SCALE, fps=fps)


def pairs_of(item):
    """[(HR, LR, frame index)] for one source, and the degradation it took."""
    if item["kind"] == "sequence":
        frames, fps = sources.load_sequence(item["path"])
        d = degradation_for(fps)
        hr, lr = degrade.degrade(frames, d)
        keep = np.unique(np.linspace(0, len(hr) - 1, FRAMES_PER_SEQUENCE).round().astype(int))
        return [(hr[i], lr[i], int(i)) for i in keep], d
    d = degradation_for(30)
    hr, lr = degrade.degrade(sources.load_image(item["path"]), d)
    return [(hr, lr, 0)], d


demo, demo_degradation = pairs_of(items[0])
hr_demo, lr_demo, _ = demo[0]
print(items[0]["dataset"], items[0]["path"].name)
print("HR", hr_demo.shape, hr_demo.dtype)
print("LR", lr_demo.shape, lr_demo.dtype)
print(demo_degradation.as_dict())

## 3. Tiling

Patches are cut on the HR grid at multiples of `SCALE`, so each one maps onto an exact LR
patch with no rounding. A patch survives only if the variance of its luma clears
`VAR_MIN`: flat background carries no information about upscaling, and on a desktop frame
it is most of the pixels.

In [ ]:
"""Cut aligned LR/HR patch pairs out of one frame, keeping the detailed ones."""

LUMA = np.array([0.299, 0.587, 0.114], dtype=np.float32)


def tile(hr, lr, rng):
    pairs = []
    for y in range(0, hr.shape[0] - PATCH_HR + 1, STRIDE_HR):
        for x in range(0, hr.shape[1] - PATCH_HR + 1, STRIDE_HR):
            y_hr, x_hr = y - y % SCALE, x - x % SCALE
            hr_patch = hr[y_hr:y_hr + PATCH_HR, x_hr:x_hr + PATCH_HR]
            if (hr_patch.astype(np.float32) @ LUMA).var() < VAR_MIN:
                continue
            y_lr, x_lr = y_hr // SCALE, x_hr // SCALE
            pairs.append((lr[y_lr:y_lr + PATCH_LR, x_lr:x_lr + PATCH_LR], hr_patch))

    if len(pairs) > MAX_PATCHES_PER_FRAME:
        keep = rng.choice(len(pairs), MAX_PATCHES_PER_FRAME, replace=False)
        pairs = [pairs[i] for i in keep]
    return pairs


print(f"{items[0]['path'].name}: {len(tile(hr_demo, lr_demo, rng))} patches kept")

In [ ]:
"""Run every source through degrade + tile, keep its split, and stack the patches."""

buckets = {split: {"lr": [], "hr": [], "frames": [], "sources": []} for split in ("train", "val")}

start = time.time()
for n, item in enumerate(items, 1):
    bucket = buckets[item["split"]]
    pairs, d = pairs_of(item)
    base = {"captures": CAPTURE_DIR, "synthetic": SYNTHETIC_DIR}.get(item["dataset"], sources.RAW / item["dataset"])
    name = item["path"].relative_to(base).as_posix()
    first = len(bucket["lr"])
    for hr, lr, index in pairs:
        patches = tile(hr, lr, rng)
        bucket["lr"].extend(pair[0] for pair in patches)
        bucket["hr"].extend(pair[1] for pair in patches)
        if patches:
            bucket["frames"].append(f"{item['dataset']}/{name}" + (f"#{index}" if item["kind"] == "sequence" else ""))
    if len(bucket["lr"]) == first:
        print(f"  skipped {item['dataset']}/{name} — no patch cleared VAR_MIN")
        continue
    # patches [first, last) of this split came from this source, through this degradation
    bucket["sources"].append({"dataset": item["dataset"], "file": name, "patches": [first, len(bucket["lr"])],
                              "degradation": d.as_dict()})
    if n % 50 == 0:
        print(f"  {n}/{len(items)} sources, {time.time() - start:.0f}s")

for split, bucket in buckets.items():
    bucket["lr"] = (np.stack(bucket["lr"]) if bucket["lr"]
                    else np.empty((0, PATCH_LR, PATCH_LR, 3), np.uint8))
    bucket["hr"] = (np.stack(bucket["hr"]) if bucket["hr"]
                    else np.empty((0, PATCH_HR, PATCH_HR, 3), np.uint8))
    print(f"{split:5} {len(bucket['sources']):4} sources  {len(bucket['frames']):4} frames  "
          f"{len(bucket['lr']):6} patches  lr {bucket['lr'].shape}  hr {bucket['hr'].shape}")

print(f"degraded and tiled in {time.time() - start:.1f}s")

if len(buckets["train"]["lr"]) == 0:
    raise RuntimeError("no training patches — lower VAR_MIN or add more frames")

## 4. Writing the set

Four `.npy` arrays and a manifest. `uint8` on purpose: a float32 copy is four times the
bytes for no extra information, and the training notebook divides by 255 on the way to
the device anyway.

In [ ]:
"""Save the arrays and the manifest that describes them."""

for split, bucket in buckets.items():
    np.save(OUT_DIR / f"{split}_lr.npy", bucket["lr"])
    np.save(OUT_DIR / f"{split}_hr.npy", bucket["hr"])

if DEGRADATION == "clean":
    summary = "bicubic downscale + 4:2:0, no codec"
else:
    summary = (f"stream: {'/'.join(degrade.FILTERS)} downscale, 4:2:0 BT.709, H.264 libx264 CBR "
               f"at {degrade.BPP_RANGE[0]}-{degrade.BPP_RANGE[1]} bits per pixel (log-uniform), per source")

manifest = {
    "created": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "scale": SCALE,
    "patch_lr": PATCH_LR,
    "patch_hr": PATCH_HR,
    "stride_hr": STRIDE_HR,
    "var_min": VAR_MIN,
    "max_patches_per_frame": MAX_PATCHES_PER_FRAME,
    "seed": SEED,
    "degradation": summary,
    "datasets": DATASETS,
    "frames_per_sequence": FRAMES_PER_SEQUENCE,
    "splits": {
        split: {
            "patches": int(len(bucket["lr"])),
            "frames": sorted(bucket["frames"]),
            "lr": f"{split}_lr.npy",
            "hr": f"{split}_hr.npy",
            # which patches came from which source, and the degradation each one took
            "sources": bucket["sources"],
        }
        for split, bucket in buckets.items()
    },
}

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=4), encoding="utf-8")

written = sum(p.stat().st_size for p in OUT_DIR.glob("*.npy"))
print(f"{written / 1e6:.1f} MB written to {OUT_DIR.resolve()}")
print(json.dumps({k: v for k, v in manifest.items() if k != "splits"}, indent=4))

## 5. Read it back

The check that matters is alignment: an LR patch and its HR patch have to show the same
content. If the pipeline is off by a crop or a rounding step it shows here as a shift
between the two columns, and nothing downstream would tell you — the model would simply
train to a low ceiling.

In [ ]:
"""Reload from disk and look at the pairs."""

manifest = json.loads((OUT_DIR / "manifest.json").read_text(encoding="utf-8"))
lr = np.load(OUT_DIR / manifest["splits"]["train"]["lr"])
hr = np.load(OUT_DIR / manifest["splits"]["train"]["hr"])
print("reloaded", lr.shape, hr.shape, lr.dtype)

rows = min(4, len(lr))
picks = np.random.default_rng(SEED).choice(len(lr), rows, replace=False)

fig, axes = plt.subplots(rows, 2, figsize=(5, 2.5 * rows))
axes = np.atleast_2d(axes)
for row, index in enumerate(picks):
    axes[row][0].imshow(lr[index], interpolation="nearest")
    axes[row][0].set_title(f"LR {lr.shape[1]}px", fontsize=9)
    axes[row][1].imshow(hr[index], interpolation="nearest")
    axes[row][1].set_title(f"HR {hr.shape[1]}px", fontsize=9)
    for ax in axes[row]:
        ax.axis("off")
fig.suptitle("patch pairs — the two columns must show the same content")
fig.tight_layout()
plt.show()

In [ ]:
"""Luma variance of the kept patches, against the threshold that let them through."""

variance = (hr.astype(np.float32) @ LUMA).var(axis=(1, 2))

plt.figure(figsize=(6, 3))
plt.hist(variance, bins=40)
plt.axvline(VAR_MIN, color="red", label=f"VAR_MIN = {VAR_MIN}")
plt.xlabel("luma variance of HR patch")
plt.ylabel("patches")
plt.legend()
plt.tight_layout()
plt.show()

print(f"variance  min {variance.min():.0f}  median {np.median(variance):.0f}  max {variance.max():.0f}")
print("\nnext: upscale_dummy.ipynb")